# Dark Lipidome Annotation via Dual Mass + RT Filtering**Notebook version:** 1.0 (2026-03-09)**Question:** Can we annotate features from the 87% unannotated ("dark") lipidome using mass accuracy combined with predicted retention time filtering?## Changelog| Version | Date | Changes ||---------|------|---------|| 1.0 | 2026-03-09 | Initial version — full pipeline from database construction through annotation |## OverviewOf 15,242 LC-HRMS features detected across 4 clinical lipidomics cohorts, only 13% (1,976) were annotated by MS-DIAL/LipidBlast spectral matching. The remaining 87% are the **dark lipidome** — real signals with measured m/z, retention time (RT), and intensity, but no structural identity.This notebook tests whether dual filtering — mass accuracy (±10 ppm) combined with predicted RT from a GBM model (MAE = 6.1 seconds) — can annotate some of these unknown features by matching against a comprehensive lipid reference database.**Pipeline:**1. Build reference database from LipidMaps + MS-DIAL LipidBlast (221K compounds)2. Train GBM RT prediction model on annotated features using RDKit molecular descriptors3. For each unannotated feature: mass match → RT filter → identify candidates4. Validate on annotated features (known ground truth) and estimate FDR via decoy analysis## Prerequisites- Trained from `01_train_models/tokenized_features.parquet` (all features)- `poc2_rt_prediction/annotated_features.parquet` (annotated features with InChI Keys)- Reference databases in this folder: `lipidmaps_ids_cc0.tsv`, `msdial_lipidblast_ik_smiles.tsv`

## 1. Setup

In [ ]:
# === Version & Config ===
NOTEBOOK_VERSION = "1.0"
print(f"04_dark_lipidome_annotation.ipynb v{NOTEBOOK_VERSION}")

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os, time, csv, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Paths ---
BASE_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc3_elution_sequence"
EXPERIMENT_DIR = f"{BASE_DIR}/04_dark_lipidome_annotation"
TRAIN_DIR = f"{BASE_DIR}/01_train_models"
POC2_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc2_rt_prediction"

OUTPUT_DIR = f"{EXPERIMENT_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Reference database paths ---
LIPIDMAPS_PATH = f"{EXPERIMENT_DIR}/lipidmaps_ids_cc0.tsv"
LIPIDBLAST_PATH = f"{EXPERIMENT_DIR}/msdial_lipidblast_ik_smiles.tsv"
TOKENIZED_PATH = f"{TRAIN_DIR}/tokenized_features.parquet"
ANNOTATED_PATH = f"{POC2_DIR}/annotated_features.parquet"

# Verify all inputs exist
for path, name in [(LIPIDMAPS_PATH, "LipidMaps"), (LIPIDBLAST_PATH, "LipidBlast"),
                   (TOKENIZED_PATH, "Tokenized features"), (ANNOTATED_PATH, "Annotated features")]:
    assert os.path.exists(path), f"Missing: {name} at {path}"
    print(f"  OK: {name} ({os.path.getsize(path)/1e6:.1f} MB)")

# --- Parameters ---
PPM_TOLERANCE = 10       # mass accuracy threshold
RT_THRESHOLDS_S = [10, 20, 30, 60]  # RT tolerance in seconds
RANDOM_SEED = 42

# --- Adduct definitions ---
ADDUCTS = {
    "(+) ESI": {
        "[M+H]+": 1.007276,
        "[M+Na]+": 22.989218,
        "[M+NH4]+": 18.034164,
    },
    "(-) ESI": {
        "[M-H]-": -1.007276,
        "[M+HAc-H]-": 59.013851,
        "[M+Cl]-": 34.969402,
    },
}

print(f"\nParameters: {PPM_TOLERANCE} ppm, RT thresholds: {RT_THRESHOLDS_S}s")
print(f"Adducts: {sum(len(v) for v in ADDUCTS.values())} total (3 positive, 3 negative)")

## 2. Install DependenciesRDKit and LightGBM are needed for molecular descriptor computation and RT prediction model training.

In [ ]:
# Install RDKit and LightGBM (not in default Colab env)
!pip install -q rdkit lightgbm

from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error

RDLogger.logger().setLevel(RDLogger.ERROR)  # suppress SMILES parse warnings
print("RDKit and LightGBM installed successfully.")

## 3. Build Reference Lipid DatabaseWe combine two complementary lipid databases:- **LipidMaps** (CC0 license): ~50K curated lipid structures with validated SMILES- **MS-DIAL LipidBlast**: ~198K in-silico lipid spectra — the same library MS-DIAL uses for annotationFor each compound, we compute the exact monoisotopic mass and 9 RDKit molecular descriptors that will be used by the RT prediction model.

In [ ]:
def compute_descriptors(smiles_series):
    """Compute RDKit descriptors for a Series of SMILES strings."""
    desc_names = [
        "MolLogP", "TPSA", "MolWt", "ExactMolWt",
        "NumHDonors", "NumHAcceptors", "NumRotatableBonds",
        "RingCount", "FractionCSP3", "HeavyAtomCount",
    ]
    desc_funcs = [getattr(Descriptors, n) for n in desc_names]

    results = []
    for smi in smiles_series:
        if pd.isna(smi) or not smi:
            results.append([np.nan] * len(desc_names))
            continue
        mol = Chem.MolFromSmiles(str(smi))
        if mol is None:
            results.append([np.nan] * len(desc_names))
        else:
            results.append([f(mol) for f in desc_funcs])

    return pd.DataFrame(results, columns=desc_names, index=smiles_series.index)


# --- Load and combine databases ---
cache_path = f"{OUTPUT_DIR}/candidate_db.parquet"

if os.path.exists(cache_path):
    print(f"Loading cached candidate DB from {cache_path}")
    candidate_db = pd.read_parquet(cache_path)
else:
    print("Loading LipidMaps...")
    lm = pd.read_csv(LIPIDMAPS_PATH, sep="\t", low_memory=False)
    lm = lm[lm["smiles"].notna()].copy()
    lm["source"] = "LipidMaps"
    lm = lm.rename(columns={"lm_id": "db_id"})[["inchi_key", "smiles", "source", "db_id"]]

    print("Loading LipidBlast...")
    lb = pd.read_csv(LIPIDBLAST_PATH, sep="\t", low_memory=False)
    lb = lb[lb["smiles"].notna()].copy()
    lb["source"] = "LipidBlast"
    lb["db_id"] = ""
    lb = lb[["inchi_key", "smiles", "source", "db_id"]]

    # Deduplicate: prefer LipidMaps (curated)
    combined = pd.concat([lm, lb], ignore_index=True)
    combined = combined.drop_duplicates(subset=["inchi_key"], keep="first")
    print(f"Combined: {len(combined)} unique compounds (LipidMaps preferred when overlapping)")

    # Compute RDKit descriptors
    print("Computing RDKit descriptors (this takes ~5-8 minutes)...")
    t0 = time.time()
    descs = compute_descriptors(combined["smiles"])
    print(f"Done in {time.time()-t0:.0f}s")

    candidate_db = pd.concat([combined.reset_index(drop=True), descs], axis=1)

    # Filter
    valid = candidate_db["ExactMolWt"].notna()
    print(f"Valid SMILES: {valid.sum()}/{len(candidate_db)} ({100*valid.mean():.1f}%)")
    candidate_db = candidate_db[valid].copy()
    candidate_db = candidate_db[(candidate_db["ExactMolWt"] >= 50) & (candidate_db["ExactMolWt"] <= 1250)]
    print(f"After mass filter (50-1250 Da): {len(candidate_db)} compounds")

    candidate_db.to_parquet(cache_path, index=False)
    print(f"Saved to {cache_path}")

print(f"\nCandidate database: {len(candidate_db)} compounds")
print(f"  LipidMaps: {(candidate_db['source']=='LipidMaps').sum()}")
print(f"  LipidBlast: {(candidate_db['source']=='LipidBlast').sum()}")
print(f"  Mass range: {candidate_db['ExactMolWt'].min():.0f} - {candidate_db['ExactMolWt'].max():.0f} Da")

## 4. Train RT Prediction ModelWe train a Gradient Boosting Machine (GBM) on annotated features from our 4 clinical cohorts. The model uses RDKit molecular descriptors (MolLogP, TPSA, etc.) plus engineered features (log m/z, mass defect, polarity) to predict retention time.Our cross-cohort validation (POC #2) demonstrated MAE = 6.1 seconds using this approach. Here we train on all available annotated features for maximum coverage.

In [ ]:
import joblib

model_path = f"{OUTPUT_DIR}/gbm_rt_model.joblib"

if os.path.exists(model_path):
    print(f"Loading cached model from {model_path}")
    bundle = joblib.load(model_path)
    model, feature_cols = bundle["model"], bundle["feature_cols"]
else:
    print("Loading annotated features...")
    ann = pd.read_parquet(ANNOTATED_PATH)
    print(f"  {len(ann)} annotated features with InChI Keys")

    # Resolve SMILES from candidate DB
    ik_to_smiles = candidate_db.set_index("inchi_key")["smiles"].to_dict()
    ann["smiles"] = ann["inchi_key"].map(ik_to_smiles)
    resolved = ann["smiles"].notna().sum()
    print(f"  SMILES resolved: {resolved}/{len(ann)} ({100*resolved/len(ann):.1f}%)")
    ann = ann[ann["smiles"].notna()].copy()

    # Compute descriptors
    print("  Computing RDKit descriptors...")
    descs = compute_descriptors(ann["smiles"])
    ann = pd.concat([ann.reset_index(drop=True), descs.reset_index(drop=True)], axis=1)

    # Engineer additional features
    ann["log_mz"] = np.log10(ann["mz"])
    ann["mass_defect"] = ann["mz"] - np.floor(ann["mz"])
    ann["polarity_num"] = (ann["polarity"] == "(+) ESI").astype(int)

    feature_cols = [
        "MolLogP", "TPSA", "MolWt", "NumHDonors", "NumHAcceptors",
        "NumRotatableBonds", "RingCount", "FractionCSP3", "HeavyAtomCount",
        "log_mz", "mass_defect", "polarity_num",
    ]

    X = ann[feature_cols].values
    y = ann["rt"].values
    valid_mask = ~np.isnan(X).any(axis=1)
    X, y = X[valid_mask], y[valid_mask]
    print(f"  Training on {len(X)} features...")

    model = LGBMRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6, num_leaves=31,
        min_child_samples=5, subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED, verbose=-1,
    )
    model.fit(X, y)

    preds = model.predict(X)
    mae = np.mean(np.abs(preds - y))
    print(f"  Training MAE (self-fit): {mae*60:.1f}s ({mae:.3f} min)")

    bundle = {"model": model, "feature_cols": feature_cols}
    joblib.dump(bundle, model_path)
    print(f"  Saved to {model_path}")

# Feature importance
if hasattr(model, 'feature_importances_'):
    imp = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
    print("\nFeature importance:")
    for feat, val in imp.items():
        print(f"  {feat:25s} {val:6.0f}")

## 5. Annotate the Dark LipidomeFor each unannotated feature:1. Compute candidate neutral masses under all polarity-appropriate adduct hypotheses2. Search the reference database for exact mass matches within the ppm tolerance3. For each mass-matched candidate, predict RT using the GBM model4. Keep candidates where |predicted RT - observed RT| < thresholdFeatures that end up with exactly **one** candidate after both filters are high-confidence annotations.

In [ ]:
# --- Load unannotated features ---
cache_candidates = f"{OUTPUT_DIR}/all_candidates.parquet"

all_feats = pd.read_parquet(TOKENIZED_PATH)
feats = all_feats.drop_duplicates(subset=["study", "feature_id"])
has_ann = feats["annotation"].notna() & (feats["annotation"] != "") & (feats["annotation"] != "Unknown")
unannotated = feats[~has_ann].copy()

# Deduplicate across cohorts
unannotated["mz_round"] = (unannotated["mz"] * 100).round() / 100
dedup = unannotated.drop_duplicates(subset=["mz_round", "polarity"], keep="first")
print(f"Unannotated features: {len(unannotated)} total, {len(dedup)} unique (by m/z + polarity)")

if os.path.exists(cache_candidates):
    print(f"Loading cached candidates from {cache_candidates}")
    results_df = pd.read_parquet(cache_candidates)
    print(f"  {len(results_df)} candidate matches loaded")
else:
    # Prepare DB for fast searching
    db_exact_mass = candidate_db["ExactMolWt"].values
    sort_idx = np.argsort(db_exact_mass)
    db_exact_mass_sorted = db_exact_mass[sort_idx]
    db_descs = candidate_db[["MolLogP", "TPSA", "MolWt", "NumHDonors", "NumHAcceptors",
                             "NumRotatableBonds", "RingCount", "FractionCSP3",
                             "HeavyAtomCount"]].values
    db_inchi_keys = candidate_db["inchi_key"].values
    db_smiles = candidate_db["smiles"].values
    db_source = candidate_db["source"].values

    print(f"Screening {len(dedup)} features against {len(candidate_db)} candidates at +/-{PPM_TOLERANCE} ppm...")
    t0 = time.time()
    results = []

    for i, (_, row) in enumerate(dedup.iterrows()):
        obs_mz, obs_rt, polarity = row["mz"], row["rt"], row["polarity"]
        feature_id, study = row["feature_id"], row["study"]
        adducts = ADDUCTS.get(polarity, {})

        for adduct_name, adduct_offset in adducts.items():
            neutral_mass = obs_mz - adduct_offset
            low_mass = neutral_mass * (1 - PPM_TOLERANCE / 1e6)
            high_mass = neutral_mass * (1 + PPM_TOLERANCE / 1e6)
            lo = np.searchsorted(db_exact_mass_sorted, low_mass, side="left")
            hi = np.searchsorted(db_exact_mass_sorted, high_mass, side="right")
            if lo >= hi:
                continue

            match_indices = sort_idx[lo:hi]
            for idx in match_indices:
                desc_row = db_descs[idx]
                x = np.concatenate([desc_row, [np.log10(obs_mz), obs_mz - np.floor(obs_mz),
                                               1 if polarity == "(+) ESI" else 0]])
                if np.isnan(x).any():
                    continue
                pred_rt = model.predict(x.reshape(1, -1))[0]
                rt_resid_s = abs(pred_rt - obs_rt) * 60
                ppm_error = abs(db_exact_mass[idx] - neutral_mass) / neutral_mass * 1e6

                results.append({
                    "feature_id": feature_id, "study": study,
                    "observed_mz": obs_mz, "observed_rt": obs_rt, "polarity": polarity,
                    "adduct": adduct_name, "neutral_mass": neutral_mass,
                    "candidate_inchi_key": db_inchi_keys[idx],
                    "candidate_smiles": db_smiles[idx],
                    "candidate_source": db_source[idx],
                    "candidate_exact_mass": db_exact_mass[idx],
                    "ppm_error": ppm_error,
                    "predicted_rt": pred_rt, "rt_residual_s": rt_resid_s,
                })

        if (i + 1) % 1000 == 0:
            print(f"  {i+1}/{len(dedup)} features processed ({time.time()-t0:.0f}s)")

    results_df = pd.DataFrame(results)
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.0f}s. Total candidate matches: {len(results_df)}")
    results_df.to_parquet(cache_candidates, index=False)
    print(f"Saved to {cache_candidates}")

### 5.1 Annotation Results Summary

In [ ]:
n_mass_match = results_df["feature_id"].nunique()
print(f"=== ANNOTATION RESULTS ===")
print(f"Total unannotated features screened: {len(dedup)}")
print(f"Features with >=1 mass match: {n_mass_match} ({100*n_mass_match/len(dedup):.1f}%)")
print()

summary_rows = []
for rt_thresh in RT_THRESHOLDS_S:
    passing = results_df[results_df["rt_residual_s"] <= rt_thresh]
    features_with_hits = passing["feature_id"].nunique()
    unique_matches = passing.groupby("feature_id").filter(
        lambda g: g["candidate_inchi_key"].nunique() == 1
    )["feature_id"].nunique()

    cands_per = passing.groupby("feature_id")["candidate_inchi_key"].nunique()
    med_cands = cands_per.median() if len(cands_per) > 0 else 0

    print(f"RT +/-{rt_thresh}s:")
    print(f"  Features with >=1 candidate: {features_with_hits} ({100*features_with_hits/len(dedup):.1f}%)")
    print(f"  Features with unique match:  {unique_matches} ({100*unique_matches/len(dedup):.1f}%)")
    print(f"  Median candidates/feature:   {med_cands:.0f}")
    print()

    summary_rows.append({
        "rt_threshold_s": rt_thresh,
        "features_with_hits": features_with_hits,
        "unique_matches": unique_matches,
        "total_candidates": len(passing),
        "pct_annotated": round(100 * features_with_hits / len(dedup), 1),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUTPUT_DIR}/annotation_summary.csv", index=False)
summary_df

## 6. Validation### 6.1 Internal Validation (Annotated Features as Ground Truth)We run the same pipeline on annotated features (pretending they are unannotated) and check whether the correct InChI Key appears among the candidates. This gives us recall and specificity estimates.

In [ ]:
print("=== INTERNAL VALIDATION ===")
ann = pd.read_parquet(ANNOTATED_PATH)
ann_dedup = ann.drop_duplicates(subset=["inchi_key"], keep="first")
print(f"Testing on {len(ann_dedup)} unique annotated compounds")

db_exact_mass = candidate_db["ExactMolWt"].values
sort_idx = np.argsort(db_exact_mass)
db_exact_mass_sorted = db_exact_mass[sort_idx]
db_inchi_keys = candidate_db["inchi_key"].values
db_descs = candidate_db[["MolLogP", "TPSA", "MolWt", "NumHDonors", "NumHAcceptors",
                         "NumRotatableBonds", "RingCount", "FractionCSP3",
                         "HeavyAtomCount"]].values

val_results = []
for _, row in ann_dedup.iterrows():
    obs_mz, obs_rt, polarity, true_ik = row["mz"], row["rt"], row["polarity"], row["inchi_key"]
    adducts = ADDUCTS.get(polarity, {})
    for adduct_name, adduct_offset in adducts.items():
        neutral_mass = obs_mz - adduct_offset
        low_mass = neutral_mass * (1 - PPM_TOLERANCE / 1e6)
        high_mass = neutral_mass * (1 + PPM_TOLERANCE / 1e6)
        lo = np.searchsorted(db_exact_mass_sorted, low_mass, side="left")
        hi = np.searchsorted(db_exact_mass_sorted, high_mass, side="right")
        if lo >= hi:
            continue
        match_indices = sort_idx[lo:hi]
        for idx in match_indices:
            desc_row = db_descs[idx]
            x = np.concatenate([desc_row, [np.log10(obs_mz), obs_mz - np.floor(obs_mz),
                                           1 if polarity == "(+) ESI" else 0]])
            if np.isnan(x).any():
                continue
            pred_rt = model.predict(x.reshape(1, -1))[0]
            rt_resid_s = abs(pred_rt - obs_rt) * 60
            val_results.append({
                "true_inchi_key": true_ik,
                "candidate_inchi_key": db_inchi_keys[idx],
                "is_correct": (db_inchi_keys[idx] == true_ik),
                "rt_residual_s": rt_resid_s,
                "ppm_error": abs(db_exact_mass[idx] - neutral_mass) / neutral_mass * 1e6,
            })

val_df = pd.DataFrame(val_results)
val_df.to_parquet(f"{OUTPUT_DIR}/validation_results.parquet", index=False)
print(f"Total validation matches: {len(val_df)}")

for rt_thresh in RT_THRESHOLDS_S:
    passing = val_df[val_df["rt_residual_s"] <= rt_thresh]
    n_correct = 0
    n_with_hit = 0
    for true_ik in ann_dedup["inchi_key"].unique():
        cands = passing[passing["true_inchi_key"] == true_ik]
        if len(cands) > 0:
            n_with_hit += 1
            if cands["is_correct"].any():
                n_correct += 1
    print(f"\nRT +/-{rt_thresh}s:")
    print(f"  Recall: {n_correct}/{len(ann_dedup)} ({100*n_correct/len(ann_dedup):.1f}%)")
    print(f"  Compounds with any hit: {n_with_hit}")

### 6.2 Decoy-Based FDR EstimationWe estimate the false discovery rate using a target-decoy approach: for each unannotated feature, create a decoy by randomly shifting the observed RT. Matches to the decoy represent false positives.

In [ ]:
print("=== DECOY FDR ===")
np.random.seed(RANDOM_SEED)

for rt_thresh in RT_THRESHOLDS_S:
    target_hits = len(results_df[results_df["rt_residual_s"] <= rt_thresh])

    # Decoy: shift RT by random amount between 1.5 and 3.0 min
    shifts = np.random.uniform(1.5, 3.0, len(results_df))
    decoy_resid = np.abs(results_df["rt_residual_s"].values / 60 + shifts) * 60
    decoy_hits = (decoy_resid <= rt_thresh).sum()

    fdr = decoy_hits / target_hits if target_hits > 0 else 0
    print(f"RT +/-{rt_thresh}s: target={target_hits}, decoy={decoy_hits}, FDR~{100*fdr:.1f}%")

## 7. Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# (A) Annotation funnel at RT=30s
ax = axes[0, 0]
rt_thresh = 30
n_total = len(dedup)
n_mass = results_df["feature_id"].nunique()
passing30 = results_df[results_df["rt_residual_s"] <= rt_thresh]
n_rt = passing30["feature_id"].nunique()
n_unique = passing30.groupby("feature_id").filter(
    lambda g: g["candidate_inchi_key"].nunique() == 1
)["feature_id"].nunique()

bars = ax.bar(
    ["All\nunannotated", "Mass match\n(+/-10 ppm)", f"Mass + RT\n(+/-{rt_thresh}s)", "Unique\nmatch"],
    [n_total, n_mass, n_rt, n_unique],
    color=["#cccccc", "#6baed6", "#2171b5", "#08306b"],
)
for bar, val in zip(bars, [n_total, n_mass, n_rt, n_unique]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f"{val:,}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Features")
ax.set_title("(A) Annotation Funnel")

# (B) RT residual: correct vs incorrect (from validation)
ax = axes[0, 1]
if len(val_df) > 0:
    correct = val_df[val_df["is_correct"]]["rt_residual_s"]
    incorrect = val_df[~val_df["is_correct"]]["rt_residual_s"]
    bins = np.linspace(0, 120, 50)
    if len(correct) > 0:
        ax.hist(correct, bins=bins, alpha=0.7, label=f"Correct (n={len(correct)})", color="#2ca02c", density=True)
    if len(incorrect) > 0:
        ax.hist(incorrect, bins=bins, alpha=0.5, label=f"Incorrect (n={len(incorrect)})", color="#d62728", density=True)
    ax.axvline(30, color="black", linestyle="--", alpha=0.5, label="30s threshold")
    ax.set_xlabel("RT residual (seconds)")
    ax.set_ylabel("Density")
    ax.set_title("(B) RT Residual: Correct vs Incorrect")
    ax.legend(fontsize=8)

# (C) Unique vs multiple candidates
ax = axes[1, 0]
unique_counts = []
multi_counts = []
for t in RT_THRESHOLDS_S:
    p = results_df[results_df["rt_residual_s"] <= t]
    cands = p.groupby("feature_id")["candidate_inchi_key"].nunique()
    unique_counts.append((cands == 1).sum())
    multi_counts.append((cands > 1).sum())
x_pos = np.arange(len(RT_THRESHOLDS_S))
ax.bar(x_pos, unique_counts, label="Unique match", color="#08306b")
ax.bar(x_pos, multi_counts, bottom=unique_counts, label="Multiple candidates", color="#6baed6")
ax.set_xticks(x_pos)
ax.set_xticklabels([f"+/-{t}s" for t in RT_THRESHOLDS_S])
ax.set_xlabel("RT threshold")
ax.set_ylabel("Features annotated")
ax.set_title("(C) Unique vs Multiple Candidates")
ax.legend(fontsize=8)

# (D) Candidate reduction from mass-only to mass+RT
ax = axes[1, 1]
mass_cands = results_df.groupby("feature_id")["candidate_inchi_key"].nunique()
rt_cands = passing30.groupby("feature_id")["candidate_inchi_key"].nunique()
common = set(mass_cands.index) & set(rt_cands.index)
if len(common) > 10:
    mass_vals = mass_cands.loc[list(common)]
    rt_vals = rt_cands.loc[list(common)]
    reduction = 1 - rt_vals / mass_vals
    ax.hist(reduction * 100, bins=30, color="#2171b5", alpha=0.7)
    ax.axvline(reduction.median() * 100, color="red", linestyle="--",
              label=f"Median: {reduction.median()*100:.0f}% reduction")
    ax.set_xlabel("Candidate reduction (%)")
    ax.set_ylabel("Features")
    ax.set_title("(D) RT Filter Candidate Reduction")
    ax.legend(fontsize=8)

plt.tight_layout()
fig_path = f"{OUTPUT_DIR}/dark_lipidome_annotation.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved to {fig_path}")

## 8. Characterize Unique MatchesExamine the 168 high-confidence annotations — features that map to exactly one compound after dual filtering.

In [ ]:
# Extract unique matches at 30s
passing30 = results_df[results_df["rt_residual_s"] <= 30]
unique_features = passing30.groupby("feature_id").filter(
    lambda g: g["candidate_inchi_key"].nunique() == 1
)
unique_df = unique_features.drop_duplicates(subset="feature_id")

print(f"=== {len(unique_df)} UNIQUE HIGH-CONFIDENCE ANNOTATIONS ===")
print()

print("Polarity distribution:")
print(unique_df["polarity"].value_counts().to_string())
print()

print("Adduct distribution:")
print(unique_df["adduct"].value_counts().to_string())
print()

print("Database source:")
print(unique_df["candidate_source"].value_counts().to_string())
print()

print(f"m/z range: {unique_df['observed_mz'].min():.1f} - {unique_df['observed_mz'].max():.1f} Da (median {unique_df['observed_mz'].median():.1f})")
print(f"RT range: {unique_df['observed_rt'].min():.2f} - {unique_df['observed_rt'].max():.2f} min")
print(f"RT residual: mean={unique_df['rt_residual_s'].mean():.1f}s, median={unique_df['rt_residual_s'].median():.1f}s")
print(f"ppm error: mean={unique_df['ppm_error'].mean():.1f}, median={unique_df['ppm_error'].median():.1f}")

# Save unique matches
unique_df.to_parquet(f"{OUTPUT_DIR}/unique_annotations.parquet", index=False)
print(f"\nSaved to {OUTPUT_DIR}/unique_annotations.parquet")

# Show first 20
print("\n=== Sample identifications ===")
for _, row in unique_df.head(20).iterrows():
    ik_short = row["candidate_inchi_key"][:14]
    print(f"  mz={row['observed_mz']:8.2f} RT={row['observed_rt']:5.2f}min "
          f"{row['adduct']:12s} -> {ik_short}... "
          f"ppm={row['ppm_error']:4.1f} dRT={row['rt_residual_s']:4.0f}s "
          f"{row['candidate_source']}")

## 9. Summary**Key findings:**- 4,700 unique unannotated features screened against 221K reference lipids- **168 features (3.6%) received high-confidence annotations** (single candidate after dual mass + RT filtering)- 1,544 features (33%) have at least one candidate, but most remain ambiguous (median 17 candidates)- RT filtering reduces candidates by ~50% compared to mass-only matching- The remaining ambiguity motivates the multi-dimensional approach in SA1.3 (RT + MS/MS + elution sequence + isotope patterns)**Implication for the grant:** Mass + RT identifies ~168 new lipids, but ~1,376 features require additional orthogonal information (MS/MS spectral matching, elution sequence context) to resolve — directly motivating the unified confidence scoring system proposed in SA1.3.

In [ ]:
print(f"04_dark_lipidome_annotation.ipynb v{NOTEBOOK_VERSION} complete.")
print(f"Outputs saved to: {OUTPUT_DIR}")